Multi-armed Bandits Hyper-parameter tuning

In [5]:
import torch
import math, os
import torch.nn as nn
import torch.optim as optim
from dataclasses import dataclass
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset

Config class

In [6]:
@dataclass
class ParameterConfig:
    learning_rate: float
    batch_size: int
    hidden_layers: list[int]

In [7]:
CONFIGURATIONS: list[ParameterConfig] = [
    ParameterConfig(learning_rate=0.1, batch_size=64, hidden_layers=[128]),
    ParameterConfig(learning_rate=0.01, batch_size=128, hidden_layers=[128, 64]),
    ParameterConfig(learning_rate=0.001, batch_size=32, hidden_layers=[64]),
]

Params

In [8]:
input_dim = 28 * 28 # MNIST input dimension
output_dim = 10 # MNIST output class
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Persistent Storage Arrays

In [16]:
active_models = [None] * len(CONFIGURATIONS)
active_optimizers = [None] * len(CONFIGURATIONS)

Dynamically build the Trainable Network

In [10]:
def build_neural_network(config: ParameterConfig) -> nn.Sequential:
    layers = nn.Sequential()
    init_dim = input_dim

    for hidden_dim in config.hidden_layers:
        layers.append(nn.Linear(init_dim, hidden_dim))
        layers.append(nn.ReLU())
        init_dim = hidden_dim
    
    layers.append(nn.Linear(init_dim, output_dim))
    return layers

Testing the function

In [11]:
testmodel = build_neural_network(CONFIGURATIONS[0])
testmodel

Sequential(
  (0): Linear(in_features=784, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=10, bias=True)
)

Preparing the dataset

In [12]:
transform_pipeline = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = datasets.MNIST(root='/home/bukunmi/ml-journey/datasets/data', train=True, download=True, transform=transform_pipeline)
test_set = datasets.MNIST(root='/home/bukunmi/ml-journey/datasets/data', train=False, download=True, transform=transform_pipeline)

Bandit Selection Engine

In [13]:
class UCBHyperparameterTuning: # Upper Confidence Bound for Bandits
    def __init__(self, num_configs: int, total_budget: int, c: float = 2):
        self.K = num_configs
        self.max_t = total_budget
        self.c = c # Exploration Constant
        self.t = 0 # Global counter for total epochs
        self.N = [0] * self.K # Array tracking total epochs per config
        self.X_bar = [0] * self.K # Array tracking running means of validation accuracies
        self.scores = [0] * self.K # Array tracking latest evaluated UCB values
    
    def select_best_config_index(self) -> int:
        max_score = float("-inf")
        selected_index = 0
        log_t = math.log(self.t) if self.t > 0 else 0.0
        for i in range(self.K):
            if self.N[i] == 0:
                current_score = float("inf")
            else:
                exploitation = self.X_bar[i]
                exploration = self.c * math.sqrt(log_t / self.N[i])
                current_score = exploitation + exploration
            self.scores[i] = current_score
            if self.scores[i] > max_score:
                max_score = self.scores[i]
                selected_index = i
        return selected_index
    
    def update_state(self, chosen_index: int, reward: float):
        self.N[chosen_index] += 1
        self.t += 1

        delta = reward - self.X_bar[chosen_index]
        self.X_bar[chosen_index] += delta / self.N[chosen_index]

Trigger Training Epochs

In [19]:
def trigger_training_epoch(config_index: int, train_dataset, val_dataset, device) -> float:
    config = CONFIGURATIONS[config_index]

    if active_models[config_index] is None:
        model = build_neural_network(config).to(device)
        optimizer = optim.SGD(model.parameters(), lr=config.learning_rate)

        active_models[config_index] = model
        active_optimizers[config_index] = optimizer
    else:
        model = active_models[config_index]
        optimizer = active_optimizers[config_index]
    
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        images = images.view(images.size(0), -1)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
    
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            images = images.view(images.size(0), -1)
            logits = model(images)
            pred = torch.amax(logits.data, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
    return correct / total